### Import libraries

In [33]:
# 1. Imports
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    ENGINEERED_FEATURES,
    create_crop_features,
)

### 1. Load cleaned dataset

In [34]:
# 2. Load the cleaned dataset
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crop_yield_cleaned.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (999, 8)


,rainfall,fertilizer,temperature,nitrogen,phosphorus,potassium,yield,crop_type
0,1230,80,28,80,24,20,12.0,Corn
1,480,60,36,70,20,18,8.0,Sorghum
2,1250,75,29,78,22,19,11.0,Soybean
3,450,65,35,70,19,18,9.0,Sorghum
4,1200,80,27,79,22,19,11.0,Wheat


### 2. Feature Engineering (Domain-Specific Transformations)

### Create selected derived variables

In [35]:
# 3. Create engineered features
df_engineered = create_crop_features(df)

print("Original shape:", df.shape)
print("Engineered shape:", df_engineered.shape)

display(df_engineered.head())

Original shape: (999, 8)
Engineered shape: (999, 13)


,rainfall,fertilizer,temperature,nitrogen,phosphorus,potassium,yield,crop_type,total_npk,n_proportion,p_proportion,temperature_squared,rainfall_fertilizer_interaction
0,1230.0,80.0,28.0,80.0,24.0,20.0,12.0,Corn,124.0,0.645161,0.193548,784.0,98400.0
1,480.0,60.0,36.0,70.0,20.0,18.0,8.0,Sorghum,108.0,0.648148,0.185185,1296.0,28800.0
2,1250.0,75.0,29.0,78.0,22.0,19.0,11.0,Soybean,119.0,0.655462,0.184874,841.0,93750.0
3,450.0,65.0,35.0,70.0,19.0,18.0,9.0,Sorghum,107.0,0.654206,0.177570,1225.0,29250.0
4,1200.0,80.0,27.0,79.0,22.0,19.0,11.0,Wheat,120.0,0.658333,0.183333,729.0,96000.0


### Confirm new features

In [36]:
# 4. Inspect the engineered features
display(
    df_engineered[
        ENGINEERED_FEATURES
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
total_npk,999.0,113.227227,10.671231,87.00000,105.000000,114.000000,121.000000,139.000000
n_proportion,999.0,0.657811,0.042597,0.53913,0.629470,0.655738,0.686441,0.774775
p_proportion,999.0,0.192764,0.033126,0.12000,0.166667,0.193548,0.217742,0.287129
temperature_squared,999.0,908.563564,344.392221,400.00000,625.000000,841.000000,1156.000000,1600.000000
rainfall_fertilizer_interaction,999.0,66710.889890,26741.401694,20000.00000,45128.000000,65162.000000,88016.000000,136416.000000


### Review summary statistics

In [37]:
engineered_features = [
    "total_npk",
    "n_proportion",
    "p_proportion",
    "temperature_squared",
    "rainfall_fertilizer_interaction"
]

df_engineered[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
total_npk,999.0,113.227227,10.671231,87.00000,105.000000,114.000000,121.000000,139.000000
n_proportion,999.0,0.657811,0.042597,0.53913,0.629470,0.655738,0.686441,0.774775
p_proportion,999.0,0.192764,0.033126,0.12000,0.166667,0.193548,0.217742,0.287129
temperature_squared,999.0,908.563564,344.392221,400.00000,625.000000,841.000000,1156.000000,1600.000000
rainfall_fertilizer_interaction,999.0,66710.889890,26741.401694,20000.00000,45128.000000,65162.000000,88016.000000,136416.000000


### Save the engineered dataset

In [45]:
from src.feature_engineering import (
    ORIGINAL_FEATURES,
    ENGINEERED_FEATURES,
)

expected_columns = (
    ORIGINAL_FEATURES
    + ENGINEERED_FEATURES
    + ["yield"]
)

df_engineered = df_engineered[
    FINAL_COLUMN_ORDER
].copy()

print("Final column order:")
print(df_engineered.columns.tolist())

Final column order:
['crop_type', 'rainfall', 'temperature', 'fertilizer', 'nitrogen', 'phosphorus', 'potassium', 'total_npk', 'n_proportion', 'p_proportion', 'temperature_squared', 'rainfall_fertilizer_interaction', 'yield']


In [46]:
# 5. Save the engineered dataset
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crop_yield_engineered.csv"
)

df_engineered.to_csv(
    OUTPUT_PATH,
    index=False,
)

print(
    f"Engineered dataset saved to: {OUTPUT_PATH}"
)

Engineered dataset saved to: /Users/ronkecrown/Documents/gamma2-crop-yield-prediction/data/processed/crop_yield_engineered.csv


## Breakdown of Key Engineered Features

| Feature Category           | Engineered Column                 | Mathematical Formula                       | Agronomic Significance                                                                                            |
| -------------------------- | --------------------------------- | ------------------------------------------ | ----------------------------------------------------------------------------------------------------------------- |
| **Total Nutrient Level**   | `total_npk`                       | $N + P + K$                                | Summarises the combined recorded levels of nitrogen, phosphorus and potassium available for crop growth.          |
| **Nitrogen Composition**   | `n_proportion`                    | $\frac{N}{N+P+K}$                          | Measures the proportion of total NPK contributed by nitrogen and helps describe nutrient balance.                 |
| **Phosphorus Composition** | `p_proportion`                    | $\frac{P}{N+P+K}$                          | Measures the proportion of total NPK contributed by phosphorus and helps describe nutrient balance.               |
| **Thermal Response**       | `temperature_squared`             | $\text{Temperature}^2$                     | Allows the model to capture a possible non-linear relationship between temperature and crop yield.                |
| **Input Synergy**          | `rainfall_fertilizer_interaction` | $\text{Rainfall} \times \text{Fertilizer}$ | Captures how the relationship between fertiliser use and crop yield may vary under different rainfall conditions. |


In [47]:
print("Actual column order:")
print(saved_engineered_df.columns.tolist())

print("\nExpected column order:")
print(expected_columns)

Actual column order:
['crop_type', 'rainfall', 'temperature', 'fertilizer', 'nitrogen', 'phosphorus', 'potassium', 'total_npk', 'n_proportion', 'p_proportion', 'temperature_squared', 'rainfall_fertilizer_interaction', 'yield']

Expected column order:
['crop_type', 'rainfall', 'temperature', 'fertilizer', 'nitrogen', 'phosphorus', 'potassium', 'total_npk', 'n_proportion', 'p_proportion', 'temperature_squared', 'rainfall_fertilizer_interaction', 'yield']


In [48]:
saved_engineered_df = pd.read_csv(OUTPUT_PATH)

expected_columns = (
    ORIGINAL_FEATURES
    + ENGINEERED_FEATURES
    + ["yield"]
)

assert saved_engineered_df.shape == (999, 13), (
    f"Unexpected shape: {saved_engineered_df.shape}"
)

assert saved_engineered_df.columns.tolist() == expected_columns, (
    "The saved dataset columns are not in the expected order."
)

assert (
    saved_engineered_df[ENGINEERED_FEATURES]
    .isna()
    .sum()
    .sum()
    == 0
), "Missing values were found in the engineered features."

assert np.isfinite(
    saved_engineered_df[
        ENGINEERED_FEATURES
    ].to_numpy(dtype=float)
).all(), (
    "Infinite values were found in the engineered features."
)

pd.testing.assert_frame_equal(
    df_engineered.reset_index(drop=True),
    saved_engineered_df.reset_index(drop=True),
    check_dtype=False,
    rtol=1e-10,
    atol=1e-10,
)

print("All verification checks passed.")
print("Saved dataset shape:", saved_engineered_df.shape)

All verification checks passed.
Saved dataset shape: (999, 13)
